<a href="https://colab.research.google.com/github/IdaliaSalinas/atlas_review/blob/main/Algoritmo1_M_M_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import math

In [2]:
def simular_tiempos_mm1(lambda_tasa, mu_tasa, num_clientes):
    """
    1.- Algoritmo para simular los tiempos de llegada, tiempos de servicio
    y tiempos de fin de servicio de una línea de espera M/M/1.
    """
    tiempos_llegada = []
    tiempos_servicio = []
    tiempos_fin_servicio = []

    # 1. Generación de tiempos de llegada (Proceso de Poisson)
    tiempo_actual_llegada = 0.0
    for i in range(num_clientes):
        u_llegada = random.random()
        tiempo_entre_llegada = -math.log(u_llegada) / lambda_tasa
        tiempo_actual_llegada += tiempo_entre_llegada
        tiempos_llegada.append(tiempo_actual_llegada)

    # 2. Generación de duraciones de servicio (Exponencial mu)
    for i in range(num_clientes):
        v_servicio = random.random()
        duracion_servicio = -math.log(v_servicio) / mu_tasa
        tiempos_servicio.append(duracion_servicio)

    # 3. Cálculo de los tiempos de fin de servicio en base a la ocupación del servidor
    tiempo_fin_anterior = 0.0
    for i in range(num_clientes):
        tiempo_llegada_cliente = tiempos_llegada[i]
        duracion_servicio_cliente = tiempos_servicio[i]

        if tiempo_llegada_cliente > tiempo_fin_anterior:
            tiempo_inicio_servicio = tiempo_llegada_cliente
        else:
            tiempo_inicio_servicio = tiempo_fin_anterior

        tiempo_fin_cliente = tiempo_inicio_servicio + duracion_servicio_cliente
        tiempos_fin_servicio.append(tiempo_fin_cliente)
        tiempo_fin_anterior = tiempo_fin_cliente

    return tiempos_llegada, tiempos_servicio, tiempos_fin_servicio

In [25]:
def calcular_desglose_en_tiempo_t(t, tiempos_llegada, tiempos_servicio, tiempos_fin_servicio):
    """
    NUEVO ALGORITMO: Calcula el número de clientes en el sistema (atendidos + cola)
    y el número de clientes esperando estrictamente en la cola en el tiempo t.
    """
    clientes_en_sistema = 0
    clientes_en_cola = 0

    num_clientes = len(tiempos_llegada)
    tiempo_fin_anterior = 0.0

    for i in range(num_clientes):
        t_llegada = tiempos_llegada[i]
        t_fin = tiempos_fin_servicio[i]

        # Reconstruimos el tiempo exacto en que inició el servicio del cliente i
        if t_llegada > tiempo_fin_anterior:
            t_inicio = t_llegada
        else:
            t_inicio = tiempo_fin_anterior

        # Condición 1: ¿El cliente está actualmente dentro del sistema?
        if t_llegada <= t and t_fin > t:
            clientes_en_sistema += 1

            # Condición 2: Si está en el sistema, ¿aún no entra al servidor? (Está en cola)
            if t_inicio > t:
                clientes_en_cola += 1

        # Actualizamos el fin anterior para el siguiente ciclo
        tiempo_fin_anterior = t_fin

    return clientes_en_sistema, clientes_en_cola

In [32]:
# SECCIÓN DE PRUEBA Y DEMOSTRACIÓN DEL ALGORITMO
# =====================================================================

# Configuración de parámetros teóricos
TASA_LLEGADAS = 2.0
TASA_SERVICIOS = 3.0
TOTAL_CLIENTES = 50

# Ejecución de la simulación
llegadas, servicios, fines = simular_tiempos_mm1(TASA_LLEGADAS, TASA_SERVICIOS, TOTAL_CLIENTES)
# Impresión de la tabla cronológica
print("-" * 90)
print("CRONOLOGÍA COMPLETA DEL SISTEMA M/M/1")
print("-" * 90)
tiempo_fin_ant = 0.0
for idx in range(TOTAL_CLIENTES):
    t_ini = max(llegadas[idx], tiempo_fin_ant)
    print(f"Cliente {idx+1:02d} | Llegada: {llegadas[idx]:6.2f} | Inicio Serv: {t_ini:6.2f} | Fin Serv: {fines[idx]:6.2f}")
    tiempo_fin_ant = fines[idx]

# Fijamos el tiempo de evaluación (por ejemplo, la llegada del 5to cliente)
tiempo_evaluacion = llegadas[4]

# Invocación del nuevo algoritmo de desglose
en_sistema, en_cola = calcular_desglose_en_tiempo_t(tiempo_evaluacion, llegadas, servicios, fines)

print("\n" + "-" * 75)
print(f"ESTADO DE LA LINEA DE ESPERA EN EL TIEMPO t = {tiempo_evaluacion:.2f}")
print("-" * 75)
print(f"-> Clientes totales en el sistema (En servicio + En cola): {en_sistema}")
print(f"-> Clientes estrictamente esperando en la cola: {en_cola}")
print(f"-> Clientes siendo atendidos en el servidor en este instante: {en_sistema - en_cola}")
print("-" * 75)

------------------------------------------------------------------------------------------
CRONOLOGÍA COMPLETA DEL SISTEMA M/M/1
------------------------------------------------------------------------------------------
Cliente 01 | Llegada:   0.04 | Inicio Serv:   0.04 | Fin Serv:   0.07
Cliente 02 | Llegada:   0.36 | Inicio Serv:   0.36 | Fin Serv:   0.51
Cliente 03 | Llegada:   2.20 | Inicio Serv:   2.20 | Fin Serv:   2.20
Cliente 04 | Llegada:   3.52 | Inicio Serv:   3.52 | Fin Serv:   5.12
Cliente 05 | Llegada:   3.82 | Inicio Serv:   5.12 | Fin Serv:   6.53
Cliente 06 | Llegada:   3.83 | Inicio Serv:   6.53 | Fin Serv:   6.61
Cliente 07 | Llegada:   4.17 | Inicio Serv:   6.61 | Fin Serv:   6.86
Cliente 08 | Llegada:   5.31 | Inicio Serv:   6.86 | Fin Serv:   6.96
Cliente 09 | Llegada:   5.43 | Inicio Serv:   6.96 | Fin Serv:   7.11
Cliente 10 | Llegada:   5.52 | Inicio Serv:   7.11 | Fin Serv:   7.12
Cliente 11 | Llegada:   5.94 | Inicio Serv:   7.12 | Fin Serv:   7.27
Cliente 12

In [33]:
def calcular_clientes_tras_llegada_n(n, tiempos_llegada, tiempos_fin_servicio):
    """
    Algoritmo que arroja el número de clientes en el sistema inmediatamente
    después de la llegada del n-ésimo cliente (donde n es un número natural >= 1).
    """
    # Validación básica de rango para el número natural n
    if n < 1 or n > len(tiempos_llegada):
        return f"Error: 'n' debe ser un número natural entre 1 y {len(tiempos_llegada)}"

    # El tiempo exacto en que llega el n-ésimo cliente (indexación 0-based de Python)
    t_n = tiempos_llegada[n - 1]

    # Contamos cuántas salidas (muertes) ocurrieron estrictamente antes o en el tiempo t_n
    servicios_terminados_en_t_n = 0
    for tiempo_fin in tiempos_fin_servicio:
        if tiempo_fin <= t_n:
            servicios_terminados_en_t_n += 1

    # Inmediatamente después de esta llegada, han entrado exactamente n clientes en total
    clientes_en_sistema = n - servicios_terminados_en_t_n
    return clientes_en_sistema

In [34]:
# SECCIÓN DE PRUEBA CALCULAR CLIENTES DESPUÉS DE LLEGADA N
# =====================================================================

# Parámetros estocásticos
TASA_LLEGADAS = 2.0   # Lambda (λ)
TASA_SERVICIOS = 3.0  # Mu (μ)
TOTAL_CLIENTES = 50   # Cantidad total de clientes simulados

# 1. Ejecutar simulación global
llegadas, servicios, fines = simular_tiempos_mm1(TASA_LLEGADAS, TASA_SERVICIOS, TOTAL_CLIENTES)

# Mostrar la tabla de tiempos de control en la consola
print("-" * 90)
print("TABLA CRONOLÓGICA DE CONTROL (M/M/1)")
print("-" * 90)
for idx in range(TOTAL_CLIENTES):
    print(f"Cliente {idx+1:02d} | Llegada (t): {llegadas[idx]:6.2f} | Fin de Servicio: {fines[idx]:6.2f}")
print("-" * 90)

# 2. EVALUACIÓN DEL PUNTO 8
# Definimos el número natural 'n' que queremos inspeccionar (por ejemplo, el 6° cliente)
n_elegido = 6

# Invocar el nuevo algoritmo desarrollado
resultado_clientes = calcular_clientes_tras_llegada_n(n_elegido, llegadas, fines)

print(f"\nRESULTADO DEL PUNTO:")
print(f"-> Analizando el instante de la llegada del cliente n = {n_elegido} (Tiempo t = {llegadas[n_elegido-1]:.2f})")
print(f"-> Número de clientes en el sistema inmediatamente después de este arribo: {resultado_clientes} clientes.")
print("-" * 90)

------------------------------------------------------------------------------------------
TABLA CRONOLÓGICA DE CONTROL (M/M/1)
------------------------------------------------------------------------------------------
Cliente 01 | Llegada (t):   0.39 | Fin de Servicio:   0.63
Cliente 02 | Llegada (t):   0.42 | Fin de Servicio:   0.93
Cliente 03 | Llegada (t):   1.07 | Fin de Servicio:   1.72
Cliente 04 | Llegada (t):   1.17 | Fin de Servicio:   1.82
Cliente 05 | Llegada (t):   1.25 | Fin de Servicio:   1.83
Cliente 06 | Llegada (t):   1.93 | Fin de Servicio:   2.13
Cliente 07 | Llegada (t):   2.71 | Fin de Servicio:   2.84
Cliente 08 | Llegada (t):   3.13 | Fin de Servicio:   3.76
Cliente 09 | Llegada (t):   3.32 | Fin de Servicio:   3.85
Cliente 10 | Llegada (t):   4.58 | Fin de Servicio:   4.91
Cliente 11 | Llegada (t):   5.07 | Fin de Servicio:   5.20
Cliente 12 | Llegada (t):   5.58 | Fin de Servicio:   5.78
Cliente 13 | Llegada (t):   6.27 | Fin de Servicio:   6.82
Cliente 14 | L